# Row-K Rules: DiT Learning Analysis

**Rule family**: Binary ±1 sequences on a 6×6 grid where each row satisfies a K constraint.

**Model**: DiT-mini (6L, 6H, 384D), N=4096 training samples, 1M steps.

**Variants**:
| Exp | Rule | Description |
|---|---|---|
| rowK2 | row_k K=2 | Every row has exactly 2 ones |
| rowK3 | row_k K=3 | Every row has exactly 3 ones |
| varK{1,5} | row_variable_k K∈{1,5} | Each row independently K=1 or 5 |
| varK{3,4} | row_variable_k K∈{3,4} | Each row independently K=3 or 4 |
| varK{0,2,4,6} | row_variable_k K∈{0,2,4,6} | Each row independently even K |
| varK{3,4,5,6} | row_variable_k K∈{3,4,5,6} | Each row independently K∈{3..6} |
| globalK{1,5} | global_k K∈{1,5} | All rows same K drawn from {1,5} |
| globalK{2,4} | global_k K∈{2,4} | All rows same K drawn from {2,4} |

**Analysis sections**:
1. Training curves (loss, valid, mem, nan_ratio)
2. Valid/mem overlay — memorization onset
3. Grad norm evolution
4. Crash correlation: zoom into late-training mem rise
5. σ-loss evolution (DSM loss vs training step, per σ bin)
6. σ-loss curves at pre/post divergence checkpoints

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.colors as mcolors
from PIL import Image
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

SAVEROOT = "/n/holylfs06/LABS/kempner_fellow_binxuwang/Users/binxuwang/DL_Projects/DiffusionParityLearning"
FIGDIR   = "/n/home12/binxuwang/Github/DiffusionAttnConsistency/figures/rowK_analysis"
SAVE_FIGS = True
os.makedirs(FIGDIR, exist_ok=True)

%matplotlib inline
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['axes.spines.right'] = False
mpl.rcParams['axes.spines.top']   = False
plt.rcParams['figure.dpi'] = 120

# ── Experiment registry ─────────────────────────────────────────────────────
EXPS = [
    ('rowK2',       'DiT_mini_rowK2_n6_N4096',       '#1f77b4'),
    ('rowK3',       'DiT_mini_rowK3_n6_N4096',       '#aec7e8'),
    ('varK{1,5}',   'DiT_mini_rowVarK15_n6_N4096',   '#ff7f0e'),
    ('varK{3,4}',   'DiT_mini_rowVarK34_n6_N4096',   '#ffbb78'),
    ('varK{0,2,4,6}','DiT_mini_rowVarK0246_n6_N4096','#d62728'),
    ('varK{3,4,5,6}','DiT_mini_rowVarK3456_n6_N4096','#ff9896'),
    ('globalK{1,5}','DiT_mini_globalK15_n6_N4096',   '#2ca02c'),
    ('globalK{2,4}','DiT_mini_globalK24_n6_N4096',   '#98df8a'),
]
LABELS   = [e[0] for e in EXPS]
EXP_NAMES= [e[1] for e in EXPS]
COLORS   = {e[0]: e[2] for e in EXPS}
# rule family linestyle
LS = {'rowK': '-', 'varK': '--', 'globalK': ':'}
def get_ls(label):
    for k, v in LS.items():
        if label.startswith(k): return v
    return '-'

def savefig(fig, name):
    if SAVE_FIGS:
        for ext in ['png', 'pdf']:
            fig.savefig(os.path.join(FIGDIR, f"{name}.{ext}"), dpi=150, bbox_inches='tight')

# ── EMA smoothing ───────────────────────────────────────────────────────────
def ema_smooth(arr, alpha=0.9):
    if len(arr) == 0: return arr
    s = np.zeros_like(arr, dtype=np.float32)
    s[0] = arr[0]
    for i in range(1, len(arr)):
        s[i] = (1 - alpha) * arr[i] + alpha * s[i-1]
    return s

print("Experiments:")
for lbl, exp, _ in EXPS:
    ok = os.path.isdir(os.path.join(SAVEROOT, exp))
    print(f"  {'✓' if ok else '✗'} {lbl:20s}  {exp}")

In [ ]:
# ── TensorBoard loader ───────────────────────────────────────────────────────
def load_tb(exp_name, tags=None):
    tb_dir = os.path.join(SAVEROOT, exp_name, 'tensorboard')
    ea = EventAccumulator(tb_dir, size_guidance={'scalars': 0})
    ea.Reload()
    available = ea.Tags()['scalars']
    if tags is None: tags = available
    result = {}
    for tag in tags:
        if tag in available:
            events = ea.Scalars(tag)
            result[tag] = {
                'steps': np.array([e.step  for e in events]),
                'vals':  np.array([e.value for e in events]),
            }
    return result

TAGS = ['train/loss', 'train/grad_norm',
        'eval/full_valid_ratio', 'eval/per_row_valid_ratio',
        'eval/nan_ratio', 'eval/sample_mem_ratio', 'eval/row_k_valid']

tb_data = {}
for lbl, exp, _ in EXPS:
    try:
        tb_data[lbl] = load_tb(exp, TAGS)
        last = tb_data[lbl]['train/loss']['steps'][-1]
        print(f"{lbl:20s}: loaded  (last step={last:,})")
    except Exception as e:
        print(f"{lbl:20s}: ERROR — {e}")
        tb_data[lbl] = {}

## 1. Training curves — all 8 variants

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(9, 12), sharex=True)
fig.suptitle("DiT-mini Row-K Training Curves (N=4096, 6×6 grid, 1M steps)",
             fontsize=13, fontweight='bold')

panels = [
    ('train/loss',           axes[0], 'Train loss',           True,  0.95),
    ('eval/full_valid_ratio',axes[1], 'Full sequence validity',False, 0.9),
    ('eval/sample_mem_ratio',axes[2], 'Memorization ratio',   False, 0.9),
    ('eval/nan_ratio',       axes[3], 'NaN ratio',            False, 0.9),
]

for tag, ax, title, logy, alpha in panels:
    for lbl in LABELS:
        d = tb_data[lbl].get(tag)
        if d is None: continue
        c  = COLORS[lbl]
        ls = get_ls(lbl)
        ax.plot(d['steps']+1, d['vals'], color=c, lw=0.5, alpha=0.12, ls=ls)
        ax.plot(d['steps']+1, ema_smooth(d['vals'], alpha), color=c, lw=1.5,
                ls=ls, label=lbl)
    ax.set_xscale('log')
    if logy: ax.set_yscale('log')
    ax.set_title(title, fontsize=10)
    ax.grid(alpha=0.25)
    ax.tick_params(axis='x', which='both', labelbottom=True)

axes[0].legend(loc='lower left', fontsize=7, ncol=2)
axes[-1].set_xlabel('Training step', fontsize=10)

# Legend: linestyle = rule family
from matplotlib.lines import Line2D
leg_els = [Line2D([0],[0],color='gray',ls=v,lw=1.5,label=k) for k,v in LS.items()]
axes[0].legend(loc='lower left', fontsize=7, ncol=2)
axes[3].legend(handles=leg_els, loc='upper right', fontsize=8, title='rule type')

plt.tight_layout(rect=[0,0,1,0.97])
savefig(fig, 'training_curves')
plt.show()

## 2. Valid / Mem overlay — memorization onset

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
fig.suptitle("Validity and Memorization over training — all row-K variants",
             fontsize=11, fontweight='bold')

for lbl in LABELS:
    v = tb_data[lbl].get('eval/full_valid_ratio')
    m = tb_data[lbl].get('eval/sample_mem_ratio')
    c  = COLORS[lbl]
    ls = get_ls(lbl)
    if v is not None:
        ax.plot(v['steps']+1, v['vals'], color=c, lw=0.4, alpha=0.1, ls=ls)
        ax.plot(v['steps']+1, ema_smooth(v['vals'], 0.9), color=c, lw=1.6,
                ls=ls, label=lbl)
    if m is not None:
        ax.plot(m['steps']+1, m['vals'], color=c, lw=0.4, alpha=0.1, ls=':')
        ax.plot(m['steps']+1, ema_smooth(m['vals'], 0.9), color=c, lw=1.2, ls=':')

ax.set_xscale('log')
ax.set_ylim(0, 1)
ax.set_xlabel('Training step', fontsize=10)
ax.set_ylabel('Ratio', fontsize=10)
ax.text(0.98, 0.55, 'solid = validity\ndotted = memorization',
        transform=ax.transAxes, ha='right', fontsize=8, color='gray')
ax.legend(fontsize=7, ncol=2, loc='center left')
ax.grid(alpha=0.25)
plt.tight_layout()
savefig(fig, 'valid_mem_overlay')
plt.show()

## 3. Grad norm evolution

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
fig.suptitle("Gradient norm over training — all row-K variants",
             fontsize=11, fontweight='bold')

for lbl in LABELS:
    d = tb_data[lbl].get('train/grad_norm')
    if d is None: continue
    c  = COLORS[lbl]
    ls = get_ls(lbl)
    ax.plot(d['steps']+1, d['vals'], color=c, lw=0.4, alpha=0.12, ls=ls)
    ax.plot(d['steps']+1, ema_smooth(d['vals'], 0.9), color=c, lw=1.5,
            ls=ls, label=lbl)

ax.set_xscale('log')
ax.set_xlabel('Training step', fontsize=10)
ax.set_ylabel('Gradient norm', fontsize=10)
ax.legend(fontsize=7, ncol=2)
ax.grid(alpha=0.25)
plt.tight_layout()
savefig(fig, 'grad_norm')
plt.show()

## 4. Crash correlation: late training zoom (EWM α=0.95)

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(9, 12), sharex=True)
fig.suptitle("Late-training crash correlation — loss drop, validity drop, NaN rise, mem rise",
             fontsize=12, fontweight='bold')

panels = [
    ('train/loss',           axes[0], 'Train loss',           True,  0.95),
    ('eval/full_valid_ratio',axes[1], 'Full sequence validity',False, 0.95),
    ('eval/nan_ratio',       axes[2], 'NaN ratio',            False, 0.95),
    ('eval/sample_mem_ratio',axes[3], 'Memorization ratio',   False, 0.95),
]

STEP_MIN = 1e5   # zoom into steps > 100k

for tag, ax, title, logy, alpha in panels:
    for lbl in LABELS:
        d = tb_data[lbl].get(tag)
        if d is None: continue
        mask = d['steps'] >= STEP_MIN
        if not mask.any(): continue
        steps = d['steps'][mask] + 1
        vals  = d['vals'][mask]
        c  = COLORS[lbl]
        ls = get_ls(lbl)
        ax.plot(steps, vals, color=c, lw=0.5, alpha=0.12, ls=ls)
        ax.plot(steps, ema_smooth(vals, alpha), color=c, lw=1.5, ls=ls, label=lbl)
    ax.set_xscale('log')
    if logy: ax.set_yscale('log')
    ax.set_title(title, fontsize=10)
    ax.grid(alpha=0.25)
    ax.tick_params(axis='x', which='both', labelbottom=True)

axes[0].legend(loc='lower left', fontsize=7, ncol=2)
axes[-1].set_xlabel('Training step', fontsize=10)
plt.tight_layout(rect=[0,0,1,0.97])
savefig(fig, 'crash_correlation_zoom')
plt.show()

## 5. σ-loss evolution (DSM loss vs training step, per σ bin)

In [ ]:
# Add script dir to path so we can import directly
sys.path.insert(0, '/n/home12/binxuwang/Github/DiffusionAttnConsistency/scripts')
from plot_sigma_loss_evolution import (
    load_sigma_data, bin_mean, SIGMA_BINS, SIGMA_BINS_ZOOM, SPLIT_STYLES
)

# Load sigma loss records for all 8 runs
sigma_records = {}
for lbl, exp, _ in EXPS:
    try:
        sigma_records[lbl] = load_sigma_data(exp)
        print(f"{lbl:20s}: {len(sigma_records[lbl])} checkpoints")
    except Exception as e:
        print(f"{lbl:20s}: ERROR — {e}")
        sigma_records[lbl] = []

In [ ]:
# Plot σ-loss evolution: one panel per σ bin, one line set per run
n_bins = len(SIGMA_BINS)
fig, axes = plt.subplots(n_bins + 1, 1, figsize=(10, 3.2*(n_bins+1)), sharex=True)
fig.suptitle("DSM Loss vs Training Step — by σ bin (all row-K variants)",
             fontsize=12, fontweight='bold')

for b_idx, (bin_label, smin, smax) in enumerate(SIGMA_BINS):
    ax = axes[b_idx]
    for lbl in LABELS:
        recs = sigma_records[lbl]
        if not recs: continue
        steps = np.array([r['epoch'] for r in recs])
        sg    = recs[0]['sigma_grid']
        c = COLORS[lbl]; ls = get_ls(lbl)
        for split, sty in [('train','-'),('test','--'),('random',':')]:
            vals = np.array([bin_mean(r[f'loss_{split}'], sg, smin, smax) for r in recs])
            mask = ~np.isnan(vals)
            lbl_str = f"{lbl} {split}" if (b_idx == 0 and split == 'train') else None
            ax.plot((steps+1)[mask], vals[mask], color=c, lw=1.4 if split=='train' else 0.9,
                    ls=sty, alpha=1.0 if split=='train' else 0.65, label=lbl_str)
    ax.set_xscale('log')
    ax.set_title(bin_label, fontsize=10)
    ax.set_ylabel('MSE loss', fontsize=8)
    ax.grid(alpha=0.25)
    ax.tick_params(axis='x', which='both', labelbottom=True)

# Gap panel: test-train per bin per run
ax_gap = axes[-1]
bin_cols = plt.cm.plasma(np.linspace(0.05, 0.9, n_bins))
for b_idx, (bin_label, smin, smax) in enumerate(SIGMA_BINS):
    for lbl in LABELS:
        recs = sigma_records[lbl]
        if not recs: continue
        steps = np.array([r['epoch'] for r in recs])
        sg    = recs[0]['sigma_grid']
        tv = np.array([bin_mean(r['loss_train'], sg, smin, smax) for r in recs])
        te = np.array([bin_mean(r['loss_test'],  sg, smin, smax) for r in recs])
        gap  = te - tv
        mask = ~np.isnan(gap)
        ax_gap.plot((steps+1)[mask], gap[mask], color=bin_cols[b_idx],
                    lw=1.2, ls=get_ls(lbl), alpha=0.7,
                    label=bin_label if lbl == LABELS[0] else None)

ax_gap.set_xscale('log')
ax_gap.set_xlabel('Training step', fontsize=10)
ax_gap.set_ylabel('Test − Train gap', fontsize=9)
ax_gap.set_title('Generalization gap per σ bin', fontsize=10)
ax_gap.legend(fontsize=7, ncol=2, loc='upper left')
ax_gap.axhline(0, color='k', lw=0.7, ls=':')
ax_gap.grid(alpha=0.25)

axes[0].legend(fontsize=6, ncol=3, loc='upper right')
plt.tight_layout(rect=[0,0,1,0.97])
savefig(fig, 'sigma_loss_evolution')
plt.show()

## 6. σ-loss curves at pre / during / post divergence checkpoints

In [ ]:
# Find divergence epoch per run: first epoch where test-train gap in σ∈[0.2,2] > threshold
THRESH = 0.02
for lbl in LABELS:
    recs = sigma_records[lbl]
    if not recs: continue
    sg = recs[0]['sigma_grid']
    gaps = np.array([
        bin_mean(r['loss_test'], sg, 0.2, 2.0) - bin_mean(r['loss_train'], sg, 0.2, 2.0)
        for r in recs
    ])
    epochs = np.array([r['epoch'] for r in recs])
    div_idx = np.where(gaps > THRESH)[0]
    if len(div_idx):
        print(f"{lbl:20s}: gap first > {THRESH} at idx={div_idx[0]}, ep={epochs[div_idx[0]]:,}  (gap={gaps[div_idx[0]]:.3f})")
    else:
        print(f"{lbl:20s}: gap never > {THRESH} (max={gaps.max():.3f})")

In [ ]:
# Show σ-loss curves at 3 time points: before divergence, at divergence, final
# Using rowK2 as the reference run (diverges around ep ~345k, idx 35)

SPLIT_COLORS = {'train': '#2166ac', 'test': '#d73027', 'random': '#555555'}
SPLIT_LS     = {'train': '-',       'test': '-',       'random': '--'}

def draw_sigma_panel(ax, rec, title):
    sigma = rec['sigma_grid']
    all_vals = []
    for split in ['train', 'test', 'random']:
        lv = rec[f'loss_{split}']
        sv = rec[f'std_{split}']
        all_vals.append(lv)
        ax.plot(sigma, lv, color=SPLIT_COLORS[split], lw=1.8,
                ls=SPLIT_LS[split], label=split)
        lo = np.maximum(lv - sv, 1e-9)
        hi = lv + sv
        ax.fill_between(sigma, lo, hi, color=SPLIT_COLORS[split], alpha=0.12)
    ax.set_xscale('log'); ax.set_yscale('log')
    all_concat = np.concatenate(all_vals)
    pos = all_concat[all_concat > 0]
    if len(pos): ax.set_ylim(pos.min()*0.5, pos.max()*3)
    ax.set_title(title, fontsize=9)
    ax.set_xlabel('σ', fontsize=8); ax.set_ylabel('MSE loss', fontsize=8)
    ax.legend(fontsize=7, loc='upper left'); ax.grid(alpha=0.25)

time_indices = [33, 35, -1]   # before / just-after / final
time_labels  = ['before divergence', 'just after', 'final']

n_runs = len(LABELS)
n_time = len(time_indices)
fig, axes = plt.subplots(n_time, n_runs, figsize=(4.0*n_runs, 3.5*n_time), squeeze=False)
fig.suptitle("DSM Loss vs σ at pre/post divergence checkpoints",
             fontsize=12, fontweight='bold')

for ti, (widx, tlbl) in enumerate(zip(time_indices, time_labels)):
    for ei, lbl in enumerate(LABELS):
        ax = axes[ti][ei]
        recs = sigma_records[lbl]
        if not recs:
            ax.set_visible(False); continue
        try:
            rec = recs[widx]
        except IndexError:
            ax.text(0.5, 0.5, 'N/A', transform=ax.transAxes, ha='center')
            continue
        draw_sigma_panel(ax, rec, f"{lbl}\n{tlbl} (ep {rec['epoch']:,})")
        if ei > 0: ax.set_ylabel('')

plt.tight_layout(rect=[0,0,1,0.97])
savefig(fig, 'sigma_curves_checkpoints')
plt.show()

## 7. Final metrics summary

In [ ]:
def final_val(lbl, tag, n_avg=5):
    d = tb_data[lbl].get(tag)
    if d is None or len(d['vals']) == 0: return np.nan
    return float(d['vals'][-n_avg:].mean())

valid_f = [final_val(l, 'eval/full_valid_ratio') for l in LABELS]
mem_f   = [final_val(l, 'eval/sample_mem_ratio') for l in LABELS]
innov_f = [v - m for v, m in zip(valid_f, mem_f)]
nan_f   = [final_val(l, 'eval/nan_ratio') for l in LABELS]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
fig.suptitle("Final metrics — all row-K variants", fontsize=12, fontweight='bold')
x = np.arange(len(LABELS))
colors = [COLORS[l] for l in LABELS]

for ax, (ys, title) in zip(axes,
        [(valid_f, 'Final validity'), (mem_f, 'Final memorization'), (innov_f, 'Innovation rate (valid−mem)')]):
    bars = ax.bar(x, ys, color=colors, edgecolor='k', lw=0.4)
    ax.set_xticks(x); ax.set_xticklabels(LABELS, rotation=35, ha='right', fontsize=8)
    ax.set_title(title, fontsize=10); ax.grid(axis='y', alpha=0.3)
    for bar, v in zip(bars, ys):
        if not np.isnan(v):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                    f'{v:.2f}', ha='center', va='bottom', fontsize=7)

plt.tight_layout()
savefig(fig, 'final_metrics')
plt.show()

print(f"{'Run':20s}  {'valid':>7}  {'mem':>7}  {'innov':>7}  {'nan':>7}")
for l, v, m, iv, n in zip(LABELS, valid_f, mem_f, innov_f, nan_f):
    print(f"{l:20s}  {v:7.3f}  {m:7.3f}  {iv:7.3f}  {n:7.4f}")